# NOI — Exploratory Data Analysis

Exploration of the daily indicator-observation extracts that shaped the design of the
**Next Observed Indicator** forecasting system. Each section ends with the modeling
decision it motivated — the features, models, and per-OpDiv structure used in
[`NextObservedIndicatorV3.0.ipynb`](NextObservedIndicatorV3.0.ipynb).

> Internal file paths are redacted (`<PATH_REMOVED_FOR_PRIVACY>`); this notebook is
> kept output-free and is included to document the analysis process.


## 1 · Setup

In [ ]:
import os
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LOOKBACK_DAYS = 100
DATA_PATH_TEMPLATE = "<PATH_REMOVED_FOR_PRIVACY>/htoc_opdiv_obs_d{date}.csv"

plt.rcParams["figure.figsize"] = (11, 4)


## 2 · Load the raw daily extracts

Same loading convention as the modeling notebook: one CSV per day, one row per
`(indicator, source, OpDiv)` with an observation count; missing days are skipped.


In [ ]:
def load_observation_files(paths):
    """Read every existing CSV in `paths`, skipping (and reporting) missing days."""
    frames = []
    for path in paths:
        if not os.path.exists(path):
            print(f"File {path} does not exist. Skipping.")
            continue
        frames.append(pd.read_csv(path))
    return frames


end_dt = datetime.today()
start_dt = end_dt - timedelta(days=LOOKBACK_DAYS)
daily_paths = [
    DATA_PATH_TEMPLATE.format(date=dt.strftime("%Y%m%d"))
    for dt in pd.date_range(start_dt, end_dt, freq="D")
]

src = pd.concat(load_observation_files(daily_paths), ignore_index=True)
src["indicator"] = src["indicator"].astype(str).str.split(" ", expand=True)[0].str.strip()
src["OpDiv"] = src["OpDiv"].astype(str).str.strip()
src = (
    src.drop(columns=["curr_date", "indicator_key"])
       .rename(columns={"obs_date": "date"})
       .assign(date=lambda d: pd.to_datetime(d["date"]))
)


## 3 · Dataset overview

In [ ]:
print(f"Rows:            {len(src):,}")
print(f"Date coverage:   {src['date'].min():%Y-%m-%d} -> {src['date'].max():%Y-%m-%d}")
print(f"Unique indicators: {src['indicator'].nunique():,}")
print(f"OpDivs:          {sorted(src['OpDiv'].unique())}")
print()
print(src.dtypes)
print()
print("Missing values per column:")
print(src.isna().sum())


In [ ]:
# Daily observation volume per OpDiv — is reporting continuous, or do some
# OpDivs have gaps in their feeds that would masquerade as indicator dormancy?
daily_volume = src.groupby(["date", "OpDiv"])["observations"].sum().unstack(fill_value=0)

daily_volume.plot(title="Daily observation volume by OpDiv", linewidth=1)
plt.ylabel("Total observations")
plt.legend(ncol=5, fontsize=8)
plt.tight_layout()
plt.show()

# Days with zero rows for an OpDiv = feed gaps, not real silence
feed_gaps = (daily_volume == 0).sum().sort_values(ascending=False)
print("Zero-volume days per OpDiv (potential feed gaps):")
print(feed_gaps)


**Decision:** volumes differ by orders of magnitude across OpDivs and a few feeds have
gap days. Observation *counts* are therefore unreliable as a cross-OpDiv signal — the
models use the binary `seen` flag per day instead of raw counts.


## 4 · Indicator activity concentration

How is activity distributed across indicators — is re-observation dominated by a small
persistent core, or spread thinly?


In [ ]:
# Distribution of active days per indicator
active_days = src.groupby("indicator")["date"].nunique().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
active_days.hist(bins=50, ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Active days per indicator (log count)")
axes[0].set_xlabel("Days observed in window")

# Concentration: share of all observation-days coming from the top X% of indicators
cumulative = active_days.cumsum() / active_days.sum()
axes[1].plot(np.linspace(0, 100, len(cumulative)), cumulative.values * 100)
axes[1].set_title("Cumulative share of activity")
axes[1].set_xlabel("Top % of indicators")
axes[1].set_ylabel("% of observation-days")
plt.tight_layout()
plt.show()

one_hit = (active_days == 1).mean()
print(f"Indicators seen exactly once in the window: {one_hit:.1%}")


**Decision:** activity is heavily concentrated — a persistent core re-appears constantly
while a long tail is seen once or twice. This split is exactly what the windowed
frequency features (`freq_1/7/14/30/45`) capture, and why the exponential rate model
(driven by 30-day frequency) stabilizes forecasts for the sparse tail.


## 5 · Recency vs. re-observation

The key forecasting question: given how long an indicator has been quiet, how likely is
it to come back? Measured empirically by holding out the final 7/14/30 days of the
window and checking who actually returned.


In [ ]:
def recency_vs_return(df, holdout_days):
    """Empirical P(indicator returns within `holdout_days`) by prior quiet-gap."""
    cutoff = df["date"].max() - pd.Timedelta(days=holdout_days)
    history, future = df[df["date"] <= cutoff], df[df["date"] > cutoff]

    last_seen = (cutoff - history.groupby("indicator")["date"].max()).dt.days
    returned = last_seen.index.isin(future["indicator"].unique())

    return (
        pd.DataFrame({"quiet_days": last_seen, "returned": returned})
        .groupby(pd.cut(last_seen, bins=[0, 1, 3, 7, 14, 30, 60, 100], right=False),
                 observed=True)["returned"]
        .agg(["mean", "count"])
    )


for h in [7, 14, 30]:
    table = recency_vs_return(src, h)
    print(f"\n--- P(return within {h} days) by days since last seen ---")
    print(table.rename(columns={"mean": "return_rate", "count": "n"}))


**Decision:** return probability decays sharply with quiet time — recency is the single
strongest signal. This makes `last_seen` the anchor feature, motivates the recency rule
baseline (`last_seen < h`), and frames the problem as *time-to-next-event*, which is why
a survival model (Weibull AFT) earns a seat in the ensemble.


## 6 · Inter-arrival gaps and burstiness

For recurring indicators, what do the gaps between sightings look like — steady drumbeat
or bursts followed by silence?


In [ ]:
def gap_stats(dates):
    """Mean/std of day-gaps between consecutive sightings of one indicator."""
    gaps = np.diff(np.sort(dates.unique())).astype("timedelta64[D]").astype(int)
    if len(gaps) < 2:
        return pd.Series({"avg_gap": np.nan, "std_gap": np.nan, "n_sightings": len(gaps) + 1})
    return pd.Series({"avg_gap": gaps.mean(), "std_gap": gaps.std(), "n_sightings": len(gaps) + 1})


gaps_df = src.groupby("indicator")["date"].apply(gap_stats).unstack().dropna()
gaps_df["burstiness"] = (gaps_df["std_gap"] - gaps_df["avg_gap"]) / (
    gaps_df["std_gap"] + gaps_df["avg_gap"]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gaps_df["avg_gap"].hist(bins=50, ax=axes[0])
axes[0].set_title("Average gap between sightings (days)")

axes[1].scatter(gaps_df["avg_gap"], gaps_df["burstiness"], s=8, alpha=0.4)
axes[1].axhline(0, color="grey", linewidth=0.8)
axes[1].set_title("Burstiness vs. average gap")
axes[1].set_xlabel("avg_gap (days)")
axes[1].set_ylabel("burstiness  (-1 periodic ... +1 bursty)")
plt.tight_layout()
plt.show()

print(f"Indicators with positive burstiness: {(gaps_df['burstiness'] > 0).mean():.1%}")


**Decision:** a large share of indicators are bursty (clustered attacks, then silence)
rather than periodic — a memoryless rate alone would systematically mis-time them.
This is why `avg_gap` and `burstiness` are features, and why the Weibull AFT model
(whose shape parameter distinguishes bursty from steady processes) complements the
Poisson-style exponential model.


## 7 · Calendar effects

In [ ]:
by_dow = (
    src.assign(dayofweek=src["date"].dt.dayofweek)
       .groupby("dayofweek")
       .agg(total_obs=("observations", "sum"), distinct_indicators=("indicator", "nunique"))
)
by_dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

by_dow.plot(kind="bar", subplots=True, layout=(1, 2), figsize=(12, 4), legend=False,
            title=["Total observations by weekday", "Distinct indicators by weekday"])
plt.tight_layout()
plt.show()


**Decision:** weekday structure exists (reporting and attacker activity both dip on
weekends), so the dense panel carries `dayofweek` / `is_weekend` calendar features —
while the headline features stay window-based so a weekend lull isn't mistaken for
dormancy.


## 8 · Do OpDivs share indicators?

Whether one model can be trained on pooled data hinges on indicators behaving the same
way across operating divisions.


In [ ]:
# How many OpDivs does each indicator appear at?
opdivs_per_indicator = src.groupby("indicator")["OpDiv"].nunique()
print(opdivs_per_indicator.value_counts().sort_index().rename("indicators"))

# For indicators shared across OpDivs: does re-observation cadence agree?
shared = opdivs_per_indicator[opdivs_per_indicator > 1].index
cadence = (
    src[src["indicator"].isin(shared)]
    .groupby(["indicator", "OpDiv"])["date"]
    .nunique()
    .unstack()
)
print("\nActive-day counts for shared indicators (per OpDiv):")
print(cadence.describe().round(1))


**Decision:** most indicators are OpDiv-specific, and even shared ones show very
different cadences per OpDiv. Pooling would let one OpDiv's patterns contaminate
another's forecasts — so the pipeline builds a dense panel and trains **per OpDiv**:
each partner's data is modeled individually, never mixed.


## 9 · Summary of findings

| Finding | Design consequence |
|---|---|
| Observation volumes vary wildly by OpDiv; feeds have gap days | Model the binary `seen` flag, not raw counts |
| Raw extracts omit silent days entirely | Build a dense `source x date x indicator` panel with explicit zeros |
| Activity concentrated in a persistent core + sparse long tail | Windowed frequency features; rate model stabilizes the tail |
| Return probability decays sharply with quiet time | `last_seen` anchor feature; recency rule baseline; survival framing |
| Widespread bursty (non-periodic) recurrence | `avg_gap` + `burstiness` features; Weibull AFT in the ensemble |
| Weekday/weekend structure in observations | Calendar features in the panel; window-based headline features |
| Indicators and cadences are OpDiv-specific | Train and forecast per OpDiv, never pooled |

No single pattern dominates — which is the empirical case for the four-model ensemble
in `NextObservedIndicatorV3.0.ipynb` rather than a single model.
